# Task 10 — Idempotency Check

**Purpose:** Prove that re-running any layer of the pipeline produces the same final state (no duplicate rows, no data drift).

**Strategy:** Each core write is `mode=overwrite`. We capture row counts & a checksum BEFORE a re-write and compare them AFTER. All counts and checksums must be identical.

| Table | Layer | Expected behaviour |
|---|---|---|
| bronze_metadata | Bronze | 10 000 rows, same xxhash after re-ingest |
| silver_metadata | Silver | Same valid-row count |
| bronze_metadata_quarantine | Silver | Same quarantine count |
| gold_certified_metadata_final | Gold | Same certified count |
| gold_quality_metrics_final | Gold | Same metrics rows |
| pipeline_run_log | Monitor | Appended — previous entries must still exist |

## Step 1 — Capture Pre-Run Snapshot

In [ ]:
from pyspark.sql.functions import xxhash64, col, concat_ws, count, expr
from datetime import datetime

EXPECTED_BRONZE_ROWS   = 10_000
EXPECTED_TOTAL_RECORDS = 10_000   # silver_valid + quarantine must equal this

def row_count(table):
    return spark.table(table).count()

def checksum(table, key_cols):
    """XOR-aggregate row hashes — order-independent, overflow-safe fingerprint."""
    df = spark.table(table)
    return (
        df.withColumn("_h", xxhash64(*[col(c) for c in key_cols]))
          .agg(expr("bit_xor(_h)").alias("cksum"))
          .collect()[0][0]
    )

snapshot_before = {
    "bronze_rows":          row_count("metadata_governance.bronze.bronze_metadata"),
    "silver_rows":          row_count("metadata_governance.silver.silver_metadata"),
    "quarantine_rows":      row_count("metadata_governance.silver.bronze_metadata_quarantine"),
    "gold_cert_rows":       row_count("metadata_governance.gold.gold_certified_metadata_final"),
    "gold_metrics_rows":    row_count("metadata_governance.gold.gold_quality_metrics_final"),
    "log_rows":             row_count("metadata_governance.default.pipeline_run_log"),
    "bronze_checksum":      checksum("metadata_governance.bronze.bronze_metadata",
                                     ["table_name", "column_name", "database_name"]),
    "silver_checksum":      checksum("metadata_governance.silver.silver_metadata",
                                     ["table_name", "column_name", "database_name"]),
    "gold_cert_checksum":   checksum("metadata_governance.gold.gold_certified_metadata_final",
                                     ["table_name", "column_name", "certification_status", "completeness_score"]),
}

print("=" * 65)
print("STEP 1 — PRE-RUN SNAPSHOT")
print("=" * 65)
for k, v in snapshot_before.items():
    print(f"  {k:<30} : {v}")
print("=" * 65)

## Step 2 — Re-Run Bronze Write (Overwrite)

Re-ingest from S3 and overwrite the Bronze table. Row count and checksum must be identical to the snapshot above.

In [ ]:
from pyspark.sql.functions import current_timestamp, current_date, lit

df_bronze = (
    spark.read
         .option("header", "true")
         .option("inferSchema", "true")
         .csv("s3://test-capstone-project-metadata/metadata_realistic_10k.csv")
         .withColumn("ingestion_timestamp", current_timestamp())
         .withColumn("load_date", current_date())
         .withColumn("record_source", lit("AWS_S3"))
)

df_bronze.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("metadata_governance.bronze.bronze_metadata")

bronze_rerun_rows = row_count("metadata_governance.bronze.bronze_metadata")
bronze_rerun_checksum = checksum(
    "metadata_governance.bronze.bronze_metadata",
    ["table_name", "column_name", "database_name"]
)

print(f"Re-run bronze rows     : {bronze_rerun_rows}")
print(f"Re-run bronze checksum : {bronze_rerun_checksum}")

assert bronze_rerun_rows == snapshot_before["bronze_rows"], \
    f"IDEMPOTENCY FAIL — bronze rows changed: {snapshot_before['bronze_rows']} → {bronze_rerun_rows}"
assert bronze_rerun_checksum == snapshot_before["bronze_checksum"], \
    "IDEMPOTENCY FAIL — bronze checksum changed (source data drift detected)"

print("PASS — Bronze layer is idempotent")

## Step 3 — Verify Silver Invariant

Re-apply the Silver validation rules without writing. Valid + quarantine must still equal 10 000.

In [ ]:
from pyspark.sql.functions import isnull, when, concat_ws

bronze_df = spark.table("metadata_governance.bronze.bronze_metadata")

df_valid_recheck = bronze_df.filter(
    col("table_name").isNotNull()    & (col("table_name")    != "") &
    col("schema_name").isNotNull()   & (col("schema_name")   != "") &
    col("database_name").isNotNull() & (col("database_name") != "")
)

df_invalid_recheck = bronze_df.filter(
    isnull(col("table_name"))    | (col("table_name")    == "") |
    isnull(col("schema_name"))   | (col("schema_name")   == "") |
    isnull(col("database_name")) | (col("database_name") == "")
)

valid_recheck   = df_valid_recheck.count()
invalid_recheck = df_invalid_recheck.count()
total_recheck   = valid_recheck + invalid_recheck

print(f"Silver valid (recheck)      : {valid_recheck}")
print(f"Quarantine (recheck)        : {invalid_recheck}")
print(f"Total                       : {total_recheck}")

assert total_recheck == EXPECTED_TOTAL_RECORDS, \
    f"IDEMPOTENCY FAIL — total record count drifted: expected {EXPECTED_TOTAL_RECORDS}, got {total_recheck}"
assert valid_recheck == snapshot_before["silver_rows"], \
    f"IDEMPOTENCY FAIL — silver valid rows changed: {snapshot_before['silver_rows']} → {valid_recheck}"
assert invalid_recheck == snapshot_before["quarantine_rows"], \
    f"IDEMPOTENCY FAIL — quarantine rows changed: {snapshot_before['quarantine_rows']} → {invalid_recheck}"

print("PASS — Silver split is stable and idempotent")

## Step 4 — Verify Gold Stability

Gold tables must have the same row count and certification checksum as the snapshot.

In [ ]:
gold_cert_rows_now    = row_count("metadata_governance.gold.gold_certified_metadata_final")
gold_metrics_rows_now = row_count("metadata_governance.gold.gold_quality_metrics_final")
gold_cert_cksum_now   = checksum(
    "metadata_governance.gold.gold_certified_metadata_final",
    ["table_name", "column_name", "certification_status", "completeness_score"]
)

print(f"Gold certified rows (now)     : {gold_cert_rows_now}")
print(f"Gold certified rows (before)  : {snapshot_before['gold_cert_rows']}")
print(f"Gold metrics rows (now)       : {gold_metrics_rows_now}")
print(f"Gold checksum (now)           : {gold_cert_cksum_now}")
print(f"Gold checksum (before)        : {snapshot_before['gold_cert_checksum']}")

assert gold_cert_rows_now == snapshot_before["gold_cert_rows"], \
    f"IDEMPOTENCY FAIL — gold certified rows changed"
assert gold_metrics_rows_now == snapshot_before["gold_metrics_rows"], \
    f"IDEMPOTENCY FAIL — gold metrics rows changed"
assert gold_cert_cksum_now == snapshot_before["gold_cert_checksum"], \
    "IDEMPOTENCY FAIL — gold certification scores drifted"

print("PASS — Gold layer is idempotent")

## Step 5 — Verify Log Append-Only Behaviour

Pipeline logging is `mode=append`. Previous entries must never be overwritten — only new rows added.

In [ ]:
log_rows_now = row_count("metadata_governance.default.pipeline_run_log")

print(f"Log rows before pipeline : {snapshot_before['log_rows']}")
print(f"Log rows now             : {log_rows_now}")

assert log_rows_now >= snapshot_before["log_rows"], \
    f"IDEMPOTENCY FAIL — pipeline_run_log rows decreased (entries deleted?): " \
    f"{snapshot_before['log_rows']} → {log_rows_now}"

print("PASS — Log is append-only; no historical entries were removed")

## Step 6 — Idempotency Summary

In [ ]:
print("=" * 65)
print("IDEMPOTENCY CHECK — FINAL SUMMARY")
print("=" * 65)
checks = [
    ("Bronze rows stable",                  bronze_rerun_rows == snapshot_before["bronze_rows"]),
    ("Bronze checksum stable",              bronze_rerun_checksum == snapshot_before["bronze_checksum"]),
    ("Silver valid-row split stable",       valid_recheck == snapshot_before["silver_rows"]),
    ("Quarantine row split stable",         invalid_recheck == snapshot_before["quarantine_rows"]),
    ("Total record invariant (=10 000)",    total_recheck == EXPECTED_TOTAL_RECORDS),
    ("Gold certified rows stable",          gold_cert_rows_now == snapshot_before["gold_cert_rows"]),
    ("Gold metrics rows stable",            gold_metrics_rows_now == snapshot_before["gold_metrics_rows"]),
    ("Gold certification scores stable",    gold_cert_cksum_now == snapshot_before["gold_cert_checksum"]),
    ("Log entries not deleted",             log_rows_now >= snapshot_before["log_rows"]),
]
all_pass = True
for label, result in checks:
    status = "PASS" if result else "FAIL"
    if not result:
        all_pass = False
    print(f"  {status}  {label}")
print("=" * 65)
if all_pass:
    print("  ALL IDEMPOTENCY CHECKS PASSED")
else:
    raise AssertionError("One or more idempotency checks failed — see above")
print("=" * 65)